# 🛠️ LangChain Mastery: Tool Calling & Agents

> **Note:** This guide covers the essential mechanics of bridging the gap between LLM reasoning and real-world actions.

---

## 💡 Core Philosophy

*   **The LLM Paradox:** LLMs have immense reasoning power but are "trapped" in a text-only environment. They cannot reach out to the internet, databases, or local files directly.
*   **The Solution:** **Tools** are the "hands and feet" of the AI. They are specialized functions that allow an LLM to interact with external systems.
*   **The Workflow:** `Creation` ➔ `Binding` ➔ `Calling` ➔ `Execution`.

---

## 🔄 The 4-Step Lifecycle

### 🟢 Step 1: Tool Binding
Registering a tool with an LLM using `.bind_tools()`. This gives the model **awareness** of available capabilities through metadata and JSON schemas.

### 🟡 Step 2: Tool Calling
The LLM **decides** a tool is needed. 
> ⚠️ **Critical Distinction:** The LLM does *not* run the tool. It only produces a **Tool Call Object** containing the function name and arguments.

### 🟠 Step 3: Tool Execution
The developer (or framework) takes the arguments, runs the actual code (Python/API), and generates a **ToolMessage**. This result must be appended to the conversation history.

### 🔵 Step 4: Final Response
The LLM reviews the entire history (`HumanMessage` + `AIMessage/ToolCall` + `ToolMessage`) and generates a coherent final answer for the user.

---

## 🚀 Advanced Concept: Injected Tool Arguments

Sometimes, a tool needs data that the LLM shouldn't (or can't) guess, like a dynamic session ID or a result from a prior calculation. 

*   **`InjectedToolArg`**: An annotation that tells the LLM: *"I need this value to run, but don't try to fill it yourself. The programmer will provide it manually during execution."* This prevents hallucinations.

---

## 🎓 Interview Cheat Sheet

| Concept | Explanation |
| :--- | :--- |
| **Agent vs. Tool Calling** | Tool calling is a *capability*; an Agent is the *orchestrator* that uses those capabilities autonomously to solve complex loops. |
| **Why not direct execution?** | **Security & Control.** Manual execution acts as a 'human-in-the-loop' layer to prevent unintended or harmful API calls. |
| **ToolMessage** | A specific message type in LangChain used to pass the output of a tool back into the LLM context. |

---

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,ToolMessage
import requests

**1. Tool Creation**

In [3]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [4]:
#test tool
print(multiply.invoke({'a':3, 'b':4}))

12


In [5]:
multiply.name

'multiply'

In [6]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [7]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

**2. Tool Binding**

In [43]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
llm.invoke("Hi")

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e791b-2f01-7662-90e2-bf1e13472042-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 37, 'total_tokens': 39, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 27}})

In [44]:
llm_with_tools = llm.bind_tools([multiply])
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e791b-3afd-7093-9b69-69828a42a7cb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [45]:
#Build messages and try to maintain the conversation
query = HumanMessage('can you multiply 3 with 1000')
messages = [query]
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [46]:
result = llm_with_tools.invoke(messages)
messages.append(result) # To maintain the conversation history


#NOTE LLM willl simply suggest to use the tool with the right arguement but it will not call the tool itself.

In [47]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 1000, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'fb2bcb28-23eb-416e-9382-f4c14f27a22c': 'Co0CAQw51scouLs7Mi3s699LrO4Pna74GHk48Yggn5uI52zpV06dssaZrUCDiLRA07snGaM4bkvCwlK6+ID4q5KndYmcT1EiuxP9B4N6mlC/Fh/hl4h4q70wGxoW3MOukJSNIGJc61VjnOdPrSd6pfVXXbmdC3b2TdXMfi2y/NyoXLdqK8EYoEwcNw3skZfFPymKTMFfQGJO6C1vKOQFkPMjpln241KlGj0gc09kAjA8qWVUvUgwaXlnoTB2ddbfVhc0fUtZIX4rdsczLUBvYID78eykkBVkP44YYyevMD1CW53TjbYD0MAqdWHmIr8OyZtakqPkIBWF/mDGgWvea/mgrfEFRKqRWrXv30RfxEI='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e791b-4023-7da2-9455-e41ce3fec78d-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': 'fb2bcb28-23eb-416e-9382-f4c14f27a22c', 'type': 'tool_call'}], invalid

In [48]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'b': 1000, 'a': 3},
 'id': 'fb2bcb28-23eb-416e-9382-f4c14f27a22c',
 'type': 'tool_call'}

In [49]:
tool_result = multiply.invoke(result.tool_calls[0])
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='fb2bcb28-23eb-416e-9382-f4c14f27a22c')

In [50]:
messages.append(tool_result)
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 1000, "a": 3}'}, '__gemini_function_call_thought_signatures__': {'fb2bcb28-23eb-416e-9382-f4c14f27a22c': 'Co0CAQw51scouLs7Mi3s699LrO4Pna74GHk48Yggn5uI52zpV06dssaZrUCDiLRA07snGaM4bkvCwlK6+ID4q5KndYmcT1EiuxP9B4N6mlC/Fh/hl4h4q70wGxoW3MOukJSNIGJc61VjnOdPrSd6pfVXXbmdC3b2TdXMfi2y/NyoXLdqK8EYoEwcNw3skZfFPymKTMFfQGJO6C1vKOQFkPMjpln241KlGj0gc09kAjA8qWVUvUgwaXlnoTB2ddbfVhc0fUtZIX4rdsczLUBvYID78eykkBVkP44YYyevMD1CW53TjbYD0MAqdWHmIr8OyZtakqPkIBWF/mDGgWvea/mgrfEFRKqRWrXv30RfxEI='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e791b-4023-7da2-9455-e41ce3fec78d-0', tool_calls=[{'name': 'multiply', 'args': {'b': 1000, 'a': 3}, 'id': 'fb2bcb28-23eb-416e-9382-f4c14f27a22c', 'type': 'tool_call'}], invalid

In [52]:
final_result = llm_with_tools.invoke(messages).content
final_result

'The product of 3 and 1000 is 3000.'

### **Currency Conversion - Application**

In [97]:
# tool create

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [98]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'},
 'conversion_rate': {'title': 'Conversion Rate', 'type': 'number'}}

In [99]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1780099202,
 'time_last_update_utc': 'Sat, 30 May 2026 00:00:02 +0000',
 'time_next_update_unix': 1780185602,
 'time_next_update_utc': 'Sun, 31 May 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.1992}

In [100]:
convert.invoke({'base_currency_value':10, 'conversion_rate':95.1992})

951.9920000000001

In [101]:
# tool binding
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [112]:
messages = [HumanMessage('What is the conversion factor between INR and USD, please based on the conversion factor can you convert 10 usd to inr')]
messages

[HumanMessage(content='What is the conversion factor between INR and USD, please based on the conversion factor can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [96]:
# Continue calling tools until the model returns a final answer
while True:
  ai_message = llm_with_tools.invoke(messages)
  messages.append(ai_message)

  if not getattr(ai_message, 'tool_calls', None):
    break

  for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
      tool_message = get_conversion_factor.invoke(tool_call)
    elif tool_call['name'] == 'convert':
      tool_message = convert.invoke(tool_call)
    else:
      raise ValueError(f"Unexpected tool call: {tool_call['name']}")
    messages.append(tool_message)

ai_message.content

[{'type': 'text',
  'text': 'The conversion factor between USD and INR is 95.1992.\n10 USD is equal to 951.992 INR.',
  'extras': {'signature': 'CqQCAQw51sc8HsxGCKJO3cJYViWqxvUORH0pihm1kihUXE2ak+U+8zzf3hsDxMp3Td1nPoa9bR2JyhR72J+aRv75CXSypiqnNVdUzKYCKqkIGLCMZl8wee3z1qRbek/SsvpIXUBfDF/Rsi6ewM3hhihKo741AliNDzd28BRhSIRN+c082V0opayyuzDhNieHSurCLA+KTrYKkuPNTZ22/HlSZRgjYnWVbmDIC0bgzzMBE/Ra7OJqCPy4w8+zdNPt9Y52znA7ZZz1b3/NdF/W9wtfKATU0G1xYEVXtpKcmhwfxZ4+K66z7b/ySbXzg//BX/gsxjz33eoAUTPKULCIIALGrYKUplW/SdCD54PcVXJBIxC9jfTafQ6B/diFIf04a6+Y4dR8cw=='}}]